## Actividad 3_20: Perros y gatos
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para enseñar a este software a diferenciar entre perros y gatos.

Para ello vamos a cargar los datos y etiquetarlos, a lanzar un Random Forest Classifier para establecer un punto de partida que debemos mejorar y después, vamos a tratar de solucionar el problema con una red neuronal convencional.
</div>

In [31]:
import tensorflow as tf
import gc
from tensorflow.keras import backend as K
import os

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Evitar que TensorFlow reserve toda la GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU lista")
print(tf.config.list_physical_devices("GPU"))
def reset_tf():
    K.clear_session()
    gc.collect()

GPU lista
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
import pandas as pd
import numpy as np
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('./PetImages/')
#Clase Kirmizi será la clase 0.0
#Clase Siirt será la clase 1.0


photos =  []
labels = []

In [4]:
#2. IMPORTAMOS LOS DATOS:
for idx,folder in enumerate(folders):
    for file in listdir('./PetImages/'+folder):
        #Cargamos la imagen.
        #load_img sirve para cargar las imágenes en memoria. Tiene distintos parámetros para modificar como se cargan las imágenes.
        photo = load_img('./PetImages/'+folder+'/' + file, target_size=(128, 128)) 
        #Convertimos la imagen a un array.
        photo = img_to_array(photo)
        #Los guardamos en las listas.
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

/home/ciabd10/anaconda3/envs/tf3060/lib/python3.10/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


0
1


In [5]:
photos = asarray(photos)
labels = asarray(labels)

photos_reshape = photos.reshape(photos.shape[0],-1) 

In [6]:
from sklearn.model_selection import train_test_split

# Datos originales
X = photos / 255.0
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Redimensionar las imágenes a un formato de vector
X_train_flat = X_train.reshape(X_train.shape[0], -1)  # Convertir cada imagen a un vector
X_test_flat = X_test.reshape(X_test.shape[0], -1)    # Lo mismo para el conjunto de prueba


In [7]:
print(X_train.shape, y_train.shape)

(19998, 128, 128, 3) (19998,)


In [30]:
# Vamos a empezar con un RandomForest para comprobar que tal se clasifican las fotos
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, bootstrap=False)
forest_clf.fit(X_train_flat, y_train)

y_pred = forest_clf.predict(X_test_flat)

from sklearn.metrics import accuracy_score
print("Accuracy", accuracy_score(y_test, y_pred))

Accuracy 0.6744


### Haciendolo con red neuronal

In [10]:
print(X_train_flat.shape)
print(y_train[:10])

(19998, 49152)
[1. 1. 0. 1. 0. 0. 0. 1. 1. 0.]


In [11]:
# Haciendo la red neuronal a partir del tratamiento de PCA
from tensorflow import keras

model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(128, 128, 3)))
# Luego metemos capas ocultas
model.add(keras.layers.Dense(128, activation="relu"))
model.add(keras.layers.Dense(64, activation="relu"))
# Luego metemos la capa de salida, que tiene 5 neuronas, una por cada clase, y función de activación softmax, que es la que se suele usar para clasificación multiclase.
model.add(keras.layers.Dense(2, activation="softmax"))

from tensorflow.keras import optimizers
sgd = optimizers.Adam(learning_rate=0.0005)

model.compile(loss="sparse_categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

2026-04-21 17:21:30.061989: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-21 17:21:30.062203: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-21 17:21:30.062312: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [12]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, validation_split=0.1, callbacks=[early_stopping_cb], batch_size=8)

Epoch 1/100


2026-04-21 17:22:03.284935: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-21 17:22:03.443744: I external/local_xla/xla/service/service.cc:168] XLA service 0x5bbfdc82a390 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-21 17:22:03.443759: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-04-21 17:22:03.537450: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-21 17:22:03.603582: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1776784923.779738  220491 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2250/2250 [==============================] - 8s 3ms/step - loss: 0.8176 - accuracy: 0.5173 - val_loss: 0.6932 - val_accuracy: 0.4935
Epoch 2/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6932 - accuracy: 0.5037 - val_loss: 0.6931 - val_accuracy: 0.5065
Epoch 3/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6933 - accuracy: 0.4994 - val_loss: 0.6932 - val_accuracy: 0.4935
Epoch 4/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6932 - accuracy: 0.4990 - val_loss: 0.6933 - val_accuracy: 0.4935
Epoch 5/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6933 - accuracy: 0.4982 - val_loss: 0.6932 - val_accuracy: 0.4935
Epoch 6/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6932 - accuracy: 0.4986 - val_loss: 0.6931 - val_accuracy: 0.5065
Epoch 7/100
2250/2250 [==============================] - 6s 3ms/step - loss: 0.6931 - accuracy: 0.5068 - val_loss: 0.6931 - val_accuracy: 0.50

In [13]:
model.evaluate(X_test, y_test)

157/157 [==============================] - 0s 2ms/step - loss: 0.6933 - accuracy: 0.4970


[0.6932662725448608, 0.4970000088214874]

In [21]:
del model_cnn
reset_tf()

## Ahora con red neuronal convolucional

In [15]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3), kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

# Capa convolucional adicional
model_cnn.add(keras.layers.Conv2D(64, (3, 3), activation="relu", kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(64, activation="relu"))
model_cnn.add(keras.layers.Dense(2, activation="softmax"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.001)
model_cnn.compile(optimizer=sgd_cnn, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [17]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20


2026-04-21 17:25:09.290278: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


563/563 [==============================] - 10s 11ms/step - loss: 0.6557 - accuracy: 0.6324 - val_loss: 0.6062 - val_accuracy: 0.6760
Epoch 2/20
563/563 [==============================] - 5s 10ms/step - loss: 0.5250 - accuracy: 0.7390 - val_loss: 0.5204 - val_accuracy: 0.7465
Epoch 3/20
563/563 [==============================] - 5s 10ms/step - loss: 0.4264 - accuracy: 0.7991 - val_loss: 0.5170 - val_accuracy: 0.7385
Epoch 4/20
563/563 [==============================] - 5s 9ms/step - loss: 0.3196 - accuracy: 0.8593 - val_loss: 0.5271 - val_accuracy: 0.7520
Epoch 5/20
563/563 [==============================] - 5s 10ms/step - loss: 0.1908 - accuracy: 0.9213 - val_loss: 0.7191 - val_accuracy: 0.7430
Epoch 6/20
563/563 [==============================] - 5s 10ms/step - loss: 0.0805 - accuracy: 0.9721 - val_loss: 1.0306 - val_accuracy: 0.7475
Epoch 7/20
563/563 [==============================] - 5s 10ms/step - loss: 0.0393 - accuracy: 0.9868 - val_loss: 1.2377 - val_accuracy: 0.7335
Epoch 8/20

In [18]:
model_cnn.evaluate(X_test, y_test)

157/157 [==============================] - 1s 5ms/step - loss: 0.4824 - accuracy: 0.7732


[0.4823632836341858, 0.7731999754905701]

In [19]:
tf.keras.backend.clear_session()

## Otra red convolucional

In [22]:
from tensorflow import keras
from tensorflow.keras import layers, optimizers, initializers

# Definición de la red convolucional
model_cnn = keras.Sequential()

# Capa de convolución: el input_shape debe ser (128, 128, 1) (1 en blanco y negro, 3 en rgb)
model_cnn.add(keras.layers.Conv2D(16, (5, 5), activation="relu", input_shape=(128, 128, 3), kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

# Capa convolucional adicional
model_cnn.add(keras.layers.Conv2D(32, (5, 5), activation="relu", kernel_initializer="glorot_uniform"))
model_cnn.add(keras.layers.MaxPooling2D((2, 2)))

model_cnn.add(keras.layers.Flatten())
model_cnn.add(keras.layers.Dense(32, activation="relu"))
model_cnn.add(keras.layers.Dense(2, activation="softmax"))

# Optimizador y compilación
sgd_cnn = optimizers.Adam(learning_rate=0.001)
model_cnn.compile(optimizer=sgd_cnn, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [23]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20
563/563 [==============================] - 7s 9ms/step - loss: 0.6629 - accuracy: 0.6088 - val_loss: 0.6284 - val_accuracy: 0.6475
Epoch 2/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5860 - accuracy: 0.6942 - val_loss: 0.5841 - val_accuracy: 0.7030
Epoch 3/20
563/563 [==============================] - 4s 7ms/step - loss: 0.4862 - accuracy: 0.7643 - val_loss: 0.5429 - val_accuracy: 0.7200
Epoch 4/20
563/563 [==============================] - 4s 7ms/step - loss: 0.3864 - accuracy: 0.8235 - val_loss: 0.5418 - val_accuracy: 0.7415
Epoch 5/20
563/563 [==============================] - 4s 7ms/step - loss: 0.2705 - accuracy: 0.8836 - val_loss: 0.6726 - val_accuracy: 0.7380
Epoch 6/20
563/563 [==============================] - 4s 7ms/step - loss: 0.1521 - accuracy: 0.9405 - val_loss: 0.8333 - val_accuracy: 0.7300
Epoch 7/20
563/563 [==============================] - 4s 7ms/step - loss: 0.0766 - accuracy: 0.9743 - val_loss: 1.0515 - val_accuracy: 0.7245
Epoch 

In [24]:
model_cnn.evaluate(X_test, y_test)

157/157 [==============================] - 1s 4ms/step - loss: 0.5076 - accuracy: 0.7668


[0.5076143145561218, 0.7667999863624573]

In [26]:
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

del model_cnn
reset_tf()

In [27]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

model_cnn = keras.Sequential([
    layers.Conv2D(16, (5,5), activation="relu", input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(32, (5,5), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(48, activation="relu",
                 kernel_regularizer=regularizers.l2(0.0001)),
    layers.Dropout(0.3),

    layers.Dense(2, activation="softmax")
])

model_cnn.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model_cnn.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[early_stopping_cb])

Epoch 1/20
563/563 [==============================] - 6s 8ms/step - loss: 0.6436 - accuracy: 0.6336 - val_loss: 0.6167 - val_accuracy: 0.7015
Epoch 2/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5608 - accuracy: 0.7301 - val_loss: 0.5626 - val_accuracy: 0.7280
Epoch 3/20
563/563 [==============================] - 4s 7ms/step - loss: 0.5035 - accuracy: 0.7758 - val_loss: 0.5447 - val_accuracy: 0.7505
Epoch 4/20
563/563 [==============================] - 4s 7ms/step - loss: 0.4577 - accuracy: 0.8125 - val_loss: 0.5303 - val_accuracy: 0.7670
Epoch 5/20
563/563 [==============================] - 4s 7ms/step - loss: 0.4106 - accuracy: 0.8433 - val_loss: 0.5479 - val_accuracy: 0.7680
Epoch 6/20
563/563 [==============================] - 4s 7ms/step - loss: 0.3566 - accuracy: 0.8754 - val_loss: 0.6210 - val_accuracy: 0.7385
Epoch 7/20
563/563 [==============================] - 4s 7ms/step - loss: 0.3240 - accuracy: 0.8962 - val_loss: 0.6355 - val_accuracy: 0.7830
Epoch 

In [29]:
model_cnn.evaluate(X_test, y_test)

157/157 [==============================] - 1s 3ms/step - loss: 0.5084 - accuracy: 0.7966


[0.508429229259491, 0.7965999841690063]